# Dashboard prototype

2022-2025년 부산신항 1-5부두의 **대기·재항**, **체류생산성**, **선석점유율**을 계산합니다.

**실행 순서:** 공통 준비 → 대기·재항 → 체류생산성 → 선석점유율

공통 준비를 먼저 실행한 뒤에는 각 분석을 따로 실행해도 됩니다.
각 분석은 원본의 복사본을 사용하며, 계산식과 분석별 필터는 기존 파일의 기준을 유지합니다.

| 결과 변수 | 내용 |
|---|---|
| `waiting_stay_monthly` | 부두·선석·월별 대기시간과 재항시간 |
| `stay_productivity_port_table` | 부두·월별 평균 체류생산성 |
| `stay_productivity_berth_table` | 부두·선석·월별 체류생산성 |
| `berth_occupancy_port_result` | 전체 기간 부두별 평균 선석점유율 |
| `berth_occupancy_berth_result` | 전체 기간 부두·선석별 평균 선석점유율 |
| `berth_occupancy_monthly_result` | 마지막 월별·부두별·선석별 선석점유율 표 |

**그래프는 모두 Plotly로 출력합니다.**
대기·재항은 부두별 그래프 안에서 선석별로 나누어 표시하고, 체류생산성 그래프는 유지합니다.
선석점유율은 기존의 부두별·선석별 막대그래프를 표시하며, **총 작업시간 그래프는 표시하지 않습니다.**

분석별 중간 변수와 그래프 변수에도 `waiting_stay_`, `stay_productivity_`,
`berth_occupancy_`를 사용합니다. 공통 원본인 `ports`, `record`, `port_month`는 수정하지 않습니다.


## 1. 공통 준비

라이브러리, 경로, 기간, 컬럼 이름, 선석 정보는 여기에서만 설정합니다.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from IPython.display import display


DATA_PATH = Path("../data")

START_YEAR, END_YEAR = 2022, 2025
START = pd.Timestamp(year=START_YEAR, month=1, day=1)
END = pd.Timestamp(year=END_YEAR + 1, month=1, day=1)  # 이 시각은 분석 기간에서 제외
PERIOD_HOURS = (END - START).total_seconds() / 3600
MONTHS = pd.date_range(START, END, freq="MS", inclusive="left")

BERTH_COL = "선석"
ARRIVAL_COL = "접안(예정)일시"
DEPARTURE_COL = "출항(예정)일시"
HANDLING_COLS = ["양하", "적하", "Shift"]
YEAR_COL, MONTH_COL = "기준년도", "월"
PORT_MONTH_COLS = [f"신항{i}부두" for i in range(1, 6)]

# 원본 선석점유율의 선석 구성을 유지합니다.
# 기록이 없는 2부두 9선석과 4부두 4선석도 결과 표에 포함합니다.
EXPECTED_BERTHS = {
    1: [1, 2, 3],
    2: [4, 5, 6, 7, 8, 9],
    3: [1, 2, 3, 4],
    4: [1, 2, 3, 4],
    5: [1, 2, 3, 4],
}
BERTH_COUNTS = {
    number: len(berths)
    for number, berths in EXPECTED_BERTHS.items()
}

# 두 선석점유율 그래프에서 같은 부두 색상을 사용합니다.
# 기존 선석점유율 그래프의 색상을 한 곳에 모았습니다.
PORT_COLORS = {
    1: "#4C78A8",
    2: "#F58518",
    3: "#54A24B",
    4: "#E45756",
    5: "#B279A2",
}


In [ ]:
def load_dataset(filename):
    # DATA_PATH 한 곳의 설정을 모든 CSV에 똑같이 적용합니다.
    path = DATA_PATH / f"{filename}.csv"
    if not path.is_file():
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {path.resolve()}")

    # 파일마다 인코딩이 다를 수 있어 순서대로 읽어 봅니다.
    for encoding in ["utf-8-sig", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            continue

    raise ValueError(f"파일 인코딩을 확인해 주세요: {path.name}")


# 각 파일은 여기에서 한 번만 읽습니다.
port1 = load_dataset("port1")
port2 = load_dataset("port2")
port3 = load_dataset("port3")
port4 = load_dataset("port4")
port5 = load_dataset("port5")
port_month = load_dataset("port_month")
record = load_dataset("record")

ports = [
    (1, port1),
    (2, port2),
    (3, port3),
    (4, port4),
    (5, port5),
]

print(f"데이터 폴더: {DATA_PATH.resolve()}")
print(f"분석 기간: {START.date()} 이상 ~ {END.date()} 미만")

### 공통 날짜 변환

세 분석에서 반복되던 접안·출항 날짜 변환만 공통으로 처리합니다.
이 단계에서는 행을 삭제하거나 분석 기간을 적용하지 않습니다.

| 분석 | 기간 기준 | 중복·이상값 처리 |
|---|---|---|
| 대기·재항 | `record`의 입항일시가 분석 기간에 포함 | 원본처럼 완전 중복 제거, 출항 < 입항 제외 |
| 체류생산성 | 접안일시가 분석 기간에 포함 | 원본처럼 중복·음수 물동량·처리량 0 이하·체류시간 0 이하 제외 |
| 선석점유율 | 접안~출항 구간이 분석 기간과 겹침 | 원본처럼 시간·선석 오류 제외. 중복 제거는 별도로 적용하지 않음 |

서로 다른 분석의 필터를 하나로 합치면 결과가 달라지므로 **필터는 각 분석 안에서 유지**합니다.

In [ ]:
# 원본 ports와 별도로, 날짜만 변환한 공통 복사본을 만듭니다.
base_ports = []

for base_number, base_raw in ports:
    base_data = base_raw.copy()
    base_data.columns = base_data.columns.str.strip()

    # format="mixed": 초가 있는 날짜와 없는 날짜 등을 함께 변환합니다.
    # 원본 날짜 컬럼은 남기고 분석용 날짜 컬럼을 추가합니다.
    base_data["접안시각"] = pd.to_datetime(
        base_data[ARRIVAL_COL], format="mixed", errors="coerce"
    )
    base_data["선석종료시각"] = pd.to_datetime(
        base_data[DEPARTURE_COL], format="mixed", errors="coerce"
    )
    base_ports.append((base_number, base_data))

## 2. 대기·재항 — 부두·선석별 월별 결과

기존 계산처럼 **부두와 선명**이 같고 출항시각 차이가 24시간 이내인 접안 기록을 찾습니다.
선석은 연결 조건이 아니며, 결과의 선석은 `record`의 계선장소에서 추출한 선석입니다.

표에는 전체 입항 자료 기준 재항시간과 연결 표본 기준 재항시간을 모두 남깁니다.
그래프는 같은 표본끼리 비교하도록 **`평균대기시간`과 `연결표본_평균재항시간`**만 사용합니다.


In [ ]:
# 1. 공통 복사본을 다시 복사하여 대기·재항 전용 표를 만듭니다.
waiting_stay_port_list = []

for waiting_stay_number, waiting_stay_raw in base_ports:
    waiting_stay_data = waiting_stay_raw.copy()
    waiting_stay_data = waiting_stay_data.rename(columns={
        "선박명": "선명",  # 3부두의 선박명 컬럼을 다른 부두와 통일
        BERTH_COL: "접안자료선석",
    })
    waiting_stay_data["부두"] = waiting_stay_number
    waiting_stay_port_list.append(waiting_stay_data)

waiting_stay_port = pd.concat(waiting_stay_port_list, ignore_index=True)
waiting_stay_port = waiting_stay_port.drop_duplicates().reset_index(drop=True)
waiting_stay_port["접안ID"] = range(len(waiting_stay_port))

# 기존 대기·재항 계산과 같이 record의 완전 중복을 제거합니다.
# record 자체를 수정하지 않고 전용 복사본에서만 처리합니다.
waiting_stay_record = record.drop_duplicates().copy()

waiting_stay_record["입항시각"] = pd.to_datetime(
    waiting_stay_record["입항일시"], format="mixed", errors="coerce"
)
waiting_stay_record["출항시각"] = pd.to_datetime(
    waiting_stay_record["출항일시"], format="mixed", errors="coerce"
)

# 2. 계선장소에서 부두 번호와 선석 번호를 추출합니다.
waiting_stay_record[["부두", "선석"]] = (
    waiting_stay_record["계선장소"]
    .str.extract(r"신항\s*([1-5])\s*부두\s*(\d+)\s*선석")
    .apply(pd.to_numeric)
)

# 3. 선명의 공백과 대소문자를 통일합니다.
for waiting_stay_df in [waiting_stay_port, waiting_stay_record]:
    waiting_stay_df["연결선명"] = (
        waiting_stay_df["선명"].str.upper()
        .str.replace(r"\s+", "", regex=True)
    )

waiting_stay_port = waiting_stay_port.dropna(
    subset=["연결선명", "접안시각", "선석종료시각"]
).copy()
waiting_stay_record = waiting_stay_record.dropna(
    subset=["부두", "선석", "연결선명", "입항시각", "출항시각"]
).copy()

waiting_stay_port = waiting_stay_port.loc[
    waiting_stay_port["선석종료시각"] >= waiting_stay_port["접안시각"]
].copy()
waiting_stay_record = waiting_stay_record.loc[
    (waiting_stay_record["출항시각"] >= waiting_stay_record["입항시각"])
    & (waiting_stay_record["입항시각"] >= START)
    & (waiting_stay_record["입항시각"] < END)
].copy()

waiting_stay_record[["부두", "선석"]] = waiting_stay_record[["부두", "선석"]].astype(int)
waiting_stay_record["재항시간"] = (
    waiting_stay_record["출항시각"] - waiting_stay_record["입항시각"]
).dt.total_seconds() / 3600
waiting_stay_record["월"] = waiting_stay_record["입항시각"].dt.to_period("M").astype(str)

In [ ]:
# 4. 같은 부두·선명의 가장 가까운 출항 기록을 연결합니다.
waiting_stay_ships = pd.merge_asof(
    waiting_stay_record.sort_values("출항시각"),
    waiting_stay_port[[
        "부두", "연결선명", "접안자료선석",
        "접안시각", "선석종료시각", "접안ID",
    ]].sort_values("선석종료시각"),
    left_on="출항시각",
    right_on="선석종료시각",
    by=["부두", "연결선명"],
    direction="nearest",
    tolerance=pd.Timedelta(hours=24),
)

waiting_stay_ships["출항시각차이"] = (
    waiting_stay_ships["출항시각"] - waiting_stay_ships["선석종료시각"]
).abs()

# 입항 이후이면서 출항 이전에 접안한 기록만 대기시간 계산에 사용합니다.
waiting_stay_valid = waiting_stay_ships.loc[
    (waiting_stay_ships["접안시각"] >= waiting_stay_ships["입항시각"])
    & (waiting_stay_ships["접안시각"] <= waiting_stay_ships["출항시각"])
].copy()

# 하나의 접안 기록이 여러 입항 기록에 연결되면 시각 차이가 가장 작은 것만 남깁니다.
waiting_stay_valid = (
    waiting_stay_valid.sort_values("출항시각차이")
    .drop_duplicates("접안ID")
    .copy()
)
waiting_stay_valid["대기시간"] = (
    waiting_stay_valid["접안시각"] - waiting_stay_valid["입항시각"]
).dt.total_seconds() / 3600

# 5. 전체 입항 자료와 연결에 성공한 표본을 구분해서 집계합니다.
waiting_stay_group_cols = ["부두", "선석", "월"]
waiting_stay_monthly_all = (
    waiting_stay_record.groupby(waiting_stay_group_cols)
    .agg(
        전체입항건수=("재항시간", "size"),
        전체_평균재항시간=("재항시간", "mean"),
    )
)
waiting_stay_monthly_matched = (
    waiting_stay_valid.groupby(waiting_stay_group_cols)
    .agg(
        대기시간계산건수=("대기시간", "size"),
        평균대기시간=("대기시간", "mean"),
        연결표본_평균재항시간=("재항시간", "mean"),
    )
)

waiting_stay_monthly = waiting_stay_monthly_all.join(waiting_stay_monthly_matched)
waiting_stay_monthly["대기시간계산건수"] = waiting_stay_monthly["대기시간계산건수"].fillna(0).astype(int)
waiting_stay_monthly["분석포함률(%)"] = (
    waiting_stay_monthly["대기시간계산건수"] / waiting_stay_monthly["전체입항건수"] * 100
)
waiting_stay_monthly = waiting_stay_monthly.reset_index().sort_values(waiting_stay_group_cols).reset_index(drop=True)

with pd.option_context("display.max_rows", None, "display.precision", 10):
    display(waiting_stay_monthly)


### 대기·재항 Plotly 그래프 — 연결 표본 기준

기존 `waiting_stay_monthly` 표에서 **`평균대기시간`과 `연결표본_평균재항시간`**을 바로 사용합니다.
표의 계산과 출력은 그대로 두고, 부두별 그래프 안에서 선석별로 나누어 표시합니다.


In [ ]:
# 기존 결과 표에서 해당 부두의 행을 Plotly에 바로 전달합니다.
# y에 두 컬럼을 지정하면 각각 선으로 표시됩니다.
for waiting_stay_number in sorted(waiting_stay_monthly["부두"].unique()):
    waiting_stay_fig = px.line(
        waiting_stay_monthly.loc[
            waiting_stay_monthly["부두"] == waiting_stay_number
        ],
        x="월",
        y=["평균대기시간", "연결표본_평균재항시간"],
        # facet_col: 선석마다 나누어 그리고, 한 줄에 최대 3개를 표시합니다.
        facet_col="선석",
        facet_col_wrap=3,
        markers=True,
        labels={
            "월": "년월",
            "value": "평균시간 (시간)",
            "variable": "구분",
            "대기시간계산건수": "연결 표본 수",
        },
        hover_data={
            "대기시간계산건수": ":.0f",
            "분석포함률(%)": ":.2f",
        },
    )
    waiting_stay_fig.update_layout(
        title=f"{waiting_stay_number}부두 선석별 월별 대기·재항시간 (연결 표본)",
        template="plotly_white",
        height=500,
        legend=dict(orientation="h", y=-0.25, x=0),
        margin=dict(t=100, b=130),
    )
    waiting_stay_fig.update_xaxes(
        type="date", dtick="M12", tickformat="%Y-%m", tickangle=-45
    )
    waiting_stay_fig.update_yaxes(rangemode="tozero")
    waiting_stay_fig.update_traces(connectgaps=False)
    waiting_stay_fig.show()


## 3. 체류생산성

원본 체류생산성의 계산식을 유지합니다. 선석별 처리량 비율로 월간 천TEU를 배분한 뒤,
접안~출항 시간 합계로 나눕니다. 부두별 값은 **선석별 체류생산성의 단순평균**입니다.

대기·재항의 재항시간은 `출항−입항`, 여기의 `stay_hours`는 `출항−접안`입니다.
이 두 시간을 같은 값으로 합치지 않았습니다.

In [ ]:
# 1. 체류생산성에 필요한 전처리만 적용합니다.
def stay_productivity_prepare_data(stay_productivity_data, stay_productivity_port_name):
    stay_productivity_data = stay_productivity_data.copy()

    # 공통으로 변환한 날짜를 사용합니다.
    # 원본 체류생산성 코드와 같이 날짜 컬럼을 변환한 상태에서 중복 여부를 판단합니다.
    stay_productivity_data[ARRIVAL_COL] = stay_productivity_data["접안시각"]
    stay_productivity_data[DEPARTURE_COL] = stay_productivity_data["선석종료시각"]

    stay_productivity_data[HANDLING_COLS] = stay_productivity_data[HANDLING_COLS].apply(
        lambda stay_productivity_column: pd.to_numeric(
            stay_productivity_column.astype(str).str.replace(",", "", regex=False),
            errors="coerce",
        )
    )

    # 하나라도 물동량이 누락되면 합계도 NaN으로 두어 계산에서 제외합니다.
    stay_productivity_data["processing_teu"] = stay_productivity_data[HANDLING_COLS].sum(
        axis=1, min_count=len(HANDLING_COLS)
    )
    stay_productivity_data["stay_hours"] = (
        stay_productivity_data[DEPARTURE_COL] - stay_productivity_data[ARRIVAL_COL]
    ).dt.total_seconds() / 3600

    stay_productivity_data["year_month"] = stay_productivity_data[ARRIVAL_COL].dt.to_period("M").astype(str)
    stay_productivity_data["date"] = stay_productivity_data[ARRIVAL_COL].dt.to_period("M").dt.to_timestamp()
    stay_productivity_data["port"] = stay_productivity_port_name
    stay_productivity_data["berth"] = stay_productivity_data[BERTH_COL].astype(str)

    stay_productivity_valid = (
        stay_productivity_data[ARRIVAL_COL].dt.year.between(START_YEAR, END_YEAR)
        & stay_productivity_data[HANDLING_COLS].ge(0).all(axis=1)
        & stay_productivity_data["processing_teu"].gt(0)
        & stay_productivity_data["stay_hours"].gt(0)
        & ~stay_productivity_data.duplicated()
    )
    return stay_productivity_data.loc[stay_productivity_valid].copy()


stay_productivity_all_data = pd.concat(
    [
        stay_productivity_prepare_data(stay_productivity_raw, f"port{stay_productivity_number}")
        for stay_productivity_number, stay_productivity_raw in base_ports
    ],
    ignore_index=True,
)

# 2. 부두·월·선석별 처리량과 시간을 합산합니다.
stay_productivity_berth_result = (
    stay_productivity_all_data.groupby(
        ["port", "year_month", "date", "berth"], as_index=False
    )
    .agg(
        berth_volume=("processing_teu", "sum"),
        stay_hours=("stay_hours", "sum"),
    )
)
stay_productivity_berth_result["port_volume"] = (
    stay_productivity_berth_result.groupby(["port", "year_month"])["berth_volume"]
    .transform("sum")
)
stay_productivity_berth_result["berth_share"] = (
    stay_productivity_berth_result["berth_volume"]
    / stay_productivity_berth_result["port_volume"].replace(0, np.nan)
)

In [ ]:
# 3. 원본 port_month를 수정하지 않고 복사본을 사용합니다.
stay_productivity_port_month = port_month.copy()
stay_productivity_port_month[YEAR_COL] = pd.to_numeric(
    stay_productivity_port_month[YEAR_COL], errors="coerce"
)
stay_productivity_port_month[MONTH_COL] = pd.to_numeric(
    stay_productivity_port_month[MONTH_COL], errors="coerce"
)
stay_productivity_port_month = stay_productivity_port_month.loc[
    stay_productivity_port_month[YEAR_COL].between(START_YEAR, END_YEAR)
].copy()

# 잘못된 월을 임의로 다른 월로 바꾸거나 제외하지 않고 입력을 확인하도록 합니다.
if not stay_productivity_port_month[MONTH_COL].isin(range(1, 13)).all():
    raise ValueError("port_month의 분석 기간 내 월 값에 누락 또는 오류가 있습니다.")
if not stay_productivity_port_month[YEAR_COL].mod(1).eq(0).all():
    raise ValueError("port_month의 기준년도에 정수가 아닌 값이 있습니다.")

stay_productivity_port_month["year_month"] = (
    stay_productivity_port_month[YEAR_COL].astype(int).astype(str)
    + "-"
    + stay_productivity_port_month[MONTH_COL].astype(int).astype(str).str.zfill(2)
)

# 가로로 나뉜 부두별 물동량 컬럼을 세로 형태로 바꿉니다.
stay_productivity_port_month_long = stay_productivity_port_month.melt(
    id_vars="year_month",
    value_vars=PORT_MONTH_COLS,
    var_name="source_port",
    value_name="port_thousand_teu",
)
stay_productivity_port_month_long["port"] = stay_productivity_port_month_long["source_port"].map(
    {stay_productivity_column: f"port{stay_productivity_number}" for stay_productivity_number, stay_productivity_column in enumerate(PORT_MONTH_COLS, 1)}
)
stay_productivity_port_month_long["port_thousand_teu"] = pd.to_numeric(
    stay_productivity_port_month_long["port_thousand_teu"], errors="coerce"
)

# 같은 부두·월이 여러 행이면 조인할 때 결과가 부풀려지므로 중단합니다.
if stay_productivity_port_month_long.duplicated(["port", "year_month"]).any():
    raise ValueError("port_month에 같은 부두·월이 여러 개 있습니다. 원본을 확인해 주세요.")

# 4. 월간 천TEU를 선석별 비율로 배분하고 체류생산성을 계산합니다.
stay_productivity_berth_result = stay_productivity_berth_result.merge(
    stay_productivity_port_month_long[["port", "year_month", "port_thousand_teu"]],
    on=["port", "year_month"],
    how="left",
    validate="many_to_one",
)
stay_productivity_berth_result["estimated_thousand_teu"] = (
    stay_productivity_berth_result["port_thousand_teu"] * stay_productivity_berth_result["berth_share"]
)
stay_productivity_berth_result["stay_productivity"] = (
    stay_productivity_berth_result["estimated_thousand_teu"]
    / stay_productivity_berth_result["stay_hours"].replace(0, np.nan)
)

# 5. 원본 체류생산성 코드와 같이 선석별 체류생산성의 평균을 사용합니다.
stay_productivity_port_result = (
    stay_productivity_berth_result.groupby(["port", "year_month"], as_index=False)
    .agg(
        stay_productivity=("stay_productivity", "mean"),
        berth_count=("berth", "nunique"),
    )
)
stay_productivity_port_result["date"] = pd.to_datetime(stay_productivity_port_result["year_month"])

In [ ]:
# 6. 기존 체류생산성 그래프는 유지합니다.
stay_productivity_fig = px.line(
    stay_productivity_port_result,
    x="date", y="stay_productivity", color="port", markers=True,
    hover_data={
        "date": False, "year_month": True,
        "stay_productivity": ":.4f", "berth_count": True,
    },
)
stay_productivity_fig.update_layout(
    title="월별·부두별 평균 체류생산성",
    xaxis_title="년월",
    yaxis_title="선석별 평균 체류생산성 (천TEU/시간)",
    hovermode="x unified",
    template="plotly_white",
    legend_title="부두",
)
stay_productivity_fig.update_xaxes(dtick="M3", tickformat="%Y-%m", tickangle=-45)
stay_productivity_fig.show()

# 7. 부두별·선석별 결과 표를 만듭니다. 계산용 표는 그대로 남깁니다.
stay_productivity_port_table = stay_productivity_port_result.rename(columns={
    "port": "부두", "year_month": "년월", "berth_count": "선석수",
    "stay_productivity": "체류생산성 (천TEU/시간)",
})[["부두", "년월", "선석수", "체류생산성 (천TEU/시간)"]].copy()

stay_productivity_berth_table = stay_productivity_berth_result.rename(columns={
    "port": "부두", "year_month": "년월", "berth": "선석",
    "berth_share": "선석별배분비율",
    "estimated_thousand_teu": "선석별천TEU",
    "stay_hours": "재항시간(시간)",
    "stay_productivity": "체류생산성 (천TEU/시간)",
})[[
    "부두", "년월", "선석", "선석별배분비율", "선석별천TEU",
    "재항시간(시간)", "체류생산성 (천TEU/시간)",
]].copy()

print("부두별 결과 목록")
with pd.option_context("display.max_rows", None, "display.precision", 10):
    display(stay_productivity_port_table)
print("선석별 결과 목록")
with pd.option_context("display.max_rows", None, "display.precision", 10):
    display(stay_productivity_berth_table)


## 4. 선석점유율

기존의 **부두별 점유율**, **선석별 점유율**, **마지막 월별 결과 표**를 계산합니다.
**부두별·선석별 선석점유율 Plotly 막대그래프는 유지**하고, 총 작업시간 그래프만 제외합니다.

여기서 `작업시간_시간`은 기존 코드와 동일하게 **출항−접안 시간**입니다.
실제 하역작업 시작~완료 시간이 아닙니다. 이 시간은 점유율 계산에 필요하므로 삭제하지 않습니다.

- 접안~출항 구간을 분석 기간 또는 해당 월의 경계에서 잘라 계산합니다.
- 자료가 없는 선석·월은 `NaN`으로 남깁니다. 부두별 점유율의 분모에는 설정한 전체 선석 수를 사용합니다.
- 겹치는 기록은 합치지 않고 각 기록의 시간을 합산합니다. 따라서 100%를 넘을 수 있으며, 이를 100%로 잘라내지 않습니다.


In [ ]:
# 1. 시간과 선석만 점검합니다.
# 체류생산성의 물동량 필터를 가져오지 않습니다.
berth_occupancy_frames = []
berth_occupancy_excluded_rows = []

for berth_occupancy_number, berth_occupancy_raw in base_ports:
    berth_occupancy_data = berth_occupancy_raw.copy()
    berth_occupancy_data[BERTH_COL] = pd.to_numeric(berth_occupancy_data[BERTH_COL], errors="coerce")

    berth_occupancy_invalid = (
        berth_occupancy_data[["접안시각", "선석종료시각", BERTH_COL]].isna().any(axis=1)
        | (berth_occupancy_data["선석종료시각"] < berth_occupancy_data["접안시각"])
    )
    berth_occupancy_excluded_rows.append({
        "부두": berth_occupancy_number,
        "시간·선석 오류 또는 누락 건수": int(berth_occupancy_invalid.sum()),
    })
    berth_occupancy_data = berth_occupancy_data.loc[~berth_occupancy_invalid].copy()

    # 설정한 선석이면서 분석 기간과 겹치는 기록만 선택합니다.
    berth_occupancy_data = berth_occupancy_data.loc[
        berth_occupancy_data[BERTH_COL].isin(EXPECTED_BERTHS[berth_occupancy_number])
        & (berth_occupancy_data["접안시각"] < END)
        & (berth_occupancy_data["선석종료시각"] > START)
    ].copy()
    berth_occupancy_data[BERTH_COL] = berth_occupancy_data[BERTH_COL].astype(int)
    berth_occupancy_data["부두"] = berth_occupancy_number

    # 기간 밖의 시간은 제외합니다.
    # 예: 2021년 말 접안 후 2022년 출항 → 2022년 1월 1일부터 계산
    berth_occupancy_start = berth_occupancy_data["접안시각"].clip(lower=START)
    berth_occupancy_end = berth_occupancy_data["선석종료시각"].clip(upper=END)
    berth_occupancy_data["작업시간_시간"] = (berth_occupancy_end - berth_occupancy_start).dt.total_seconds() / 3600

    berth_occupancy_frames.append(berth_occupancy_data[[
        "부두", "선석", "접안시각", "선석종료시각", "작업시간_시간",
    ]].copy())

berth_occupancy_detail = pd.concat(berth_occupancy_frames, ignore_index=True)
berth_occupancy_excluded = pd.DataFrame(berth_occupancy_excluded_rows)

# 제외 건수는 필요할 때 berth_occupancy_excluded로 확인할 수 있습니다.
# 원본 선석점유율에 없던 중복 제거·물동량 필터는 적용하지 않습니다.

In [ ]:
# 2. 전체 기간 부두별 평균 선석점유율
berth_occupancy_port_result = (
    berth_occupancy_detail.groupby("부두")
    .agg(
        기록수=("작업시간_시간", "count"),
        총작업시간_시간=("작업시간_시간", "sum"),
    )
    .reindex(list(EXPECTED_BERTHS))
    .reset_index()
)
berth_occupancy_port_result["선석수"] = berth_occupancy_port_result["부두"].map(BERTH_COUNTS)
berth_occupancy_port_result["평균선석점유율_퍼센트"] = (
    berth_occupancy_port_result["총작업시간_시간"]
    / (berth_occupancy_port_result["선석수"] * PERIOD_HOURS) * 100
)
berth_occupancy_port_result["부두명"] = berth_occupancy_port_result["부두"].astype(str) + "부두"

print("전체 기간 부두별 평균 선석점유율")
with pd.option_context("display.max_rows", None, "display.precision", 10):
    display(berth_occupancy_port_result[[
        "부두명", "선석수", "기록수", "총작업시간_시간", "평균선석점유율_퍼센트",
    ]])

# 3. 전체 기간 부두·선석별 평균 선석점유율
berth_occupancy_expected_index = pd.MultiIndex.from_tuples(
    [
        (berth_occupancy_number, berth_occupancy_berth)
        for berth_occupancy_number, berth_occupancy_berths in EXPECTED_BERTHS.items()
        for berth_occupancy_berth in berth_occupancy_berths
    ],
    names=["부두", "선석"],
)
berth_occupancy_berth_result = (
    berth_occupancy_detail.groupby(["부두", "선석"])
    .agg(총작업시간_시간=("작업시간_시간", "sum"))
    .reindex(berth_occupancy_expected_index)
    .reset_index()
)
berth_occupancy_berth_result["평균선석점유율_퍼센트"] = (
    berth_occupancy_berth_result["총작업시간_시간"] / PERIOD_HOURS * 100
)

# reindex로 기록이 없는 선석도 표시하며, 그 값은 NaN으로 둡니다.
print("전체 기간 부두·선석별 평균 선석점유율")
with pd.option_context("display.max_rows", None, "display.precision", 10):
    display(berth_occupancy_berth_result)


### 선석점유율 Plotly 그래프

기존의 부두별·선석별 막대그래프입니다. 작업시간 자체의 그래프는 출력하지 않습니다.


In [ ]:
# 부두별 평균 선석점유율: 기존 Plotly 막대그래프를 복원합니다.
# 시간 합계가 아니라 점유율(%)을 y축에 표시합니다.
berth_occupancy_port_fig = go.Figure()

for berth_occupancy_row_index, berth_occupancy_row in berth_occupancy_port_result.iterrows():

    berth_occupancy_number = int(berth_occupancy_row["부두"])
    berth_occupancy_value = berth_occupancy_row["평균선석점유율_퍼센트"]

    berth_occupancy_port_fig.add_trace(
        go.Bar(
            x=[f"{berth_occupancy_number}부두"],
            y=[berth_occupancy_value],
            name=f"{berth_occupancy_number}부두",
            marker_color=PORT_COLORS[berth_occupancy_number],

            text=(
                f"{berth_occupancy_value:.2f}%"
                if pd.notna(berth_occupancy_value)
                else "자료 없음"
            ),
            textposition="outside",

            customdata=[[
                berth_occupancy_row["총작업시간_시간"],
                berth_occupancy_row["선석수"],
                berth_occupancy_row["기록수"]
            ]],

            hovertemplate=(
                "부두: %{x}<br>"
                "평균 선석점유율: %{y:.2f}%<br>"
                "총 작업시간: %{customdata[0]:,.1f}시간<br>"
                "선석 수: %{customdata[1]:.0f}개<br>"
                "기록 수: %{customdata[2]:,.0f}건"
                "<extra></extra>"
            )
        )
    )

# 축과 표시 범위를 설정합니다.

berth_occupancy_port_maximum = berth_occupancy_port_result["평균선석점유율_퍼센트"].max()

berth_occupancy_port_upper = (
    max(100, berth_occupancy_port_maximum * 1.15)
    if pd.notna(berth_occupancy_port_maximum)
    else 100
)

berth_occupancy_port_fig.update_layout(

    title=dict(
        text=f"{START_YEAR}~{END_YEAR}년 부산 신항 부두별 평균 선석점유율",
        x=0.5,
        xanchor="center",
        font=dict(size=20)
    ),

    xaxis=dict(
        title="부두",
        categoryorder="array",
        categoryarray=[
            "1부두",
            "2부두",
            "3부두",
            "4부두",
            "5부두"
        ],
        showgrid=False
    ),

    yaxis=dict(
        title="평균 선석점유율 (%)",
        range=[0, berth_occupancy_port_upper],
        ticksuffix="%",
        showgrid=True,
        gridcolor="#E5E7EB"
    ),

    barmode="group",
    template="plotly_white",
    autosize=True,
    height=500,
    bargap=0.35,
    showlegend=False,
    margin=dict(
        t=100,
        b=70,
        l=80,
        r=40
    )
)

# 노트북 안에 그래프를 출력합니다.

berth_occupancy_port_fig.show()


In [ ]:
# 부두·선석별 평균 선석점유율
# 시간 합계가 아니라 점유율(%)을 y축에 표시합니다.
berth_occupancy_berth_fig = go.Figure()

for berth_occupancy_number in EXPECTED_BERTHS:

    berth_occupancy_plot_data = berth_occupancy_berth_result[berth_occupancy_berth_result["부두"] == berth_occupancy_number].copy()

    berth_occupancy_labels = (
        berth_occupancy_plot_data["부두"].astype(str) + "부두 "
        + berth_occupancy_plot_data["선석"].astype(int).astype(str) + "선석"
    )

    berth_occupancy_values = berth_occupancy_plot_data["평균선석점유율_퍼센트"]

    berth_occupancy_berth_fig.add_trace(
        go.Bar(
            x=berth_occupancy_labels,
            y=berth_occupancy_values,
            name=f"{berth_occupancy_number}부두",
            marker_color=PORT_COLORS[berth_occupancy_number],
            text=[
                f"{berth_occupancy_value:.2f}%" if pd.notna(berth_occupancy_value) else "자료 없음"
                for berth_occupancy_value in berth_occupancy_values
            ],
            textposition="outside",
            customdata=berth_occupancy_plot_data[["총작업시간_시간"]].to_numpy(),
            hovertemplate=(
                "선석: %{x}<br>"
                "평균 선석점유율: %{y:.2f}%<br>"
                "총 작업시간: %{customdata[0]:,.1f}시간"
                "<extra></extra>"
            )
        )
    )

# 축과 표시 범위를 설정합니다.

berth_occupancy_berth_maximum = berth_occupancy_berth_result["평균선석점유율_퍼센트"].max()

berth_occupancy_berth_upper = (
    max(100, berth_occupancy_berth_maximum * 1.15)
    if pd.notna(berth_occupancy_berth_maximum)
    else 100
)

berth_occupancy_berth_fig.update_layout(
    title=dict(
        text=f"{START_YEAR}~{END_YEAR}년 부두별·선석별 평균 선석점유율",
        x=0.5,
        font=dict(size=20)
    ),
    xaxis=dict(
        title="부두 / 선석",
        tickangle=-45
    ),
    yaxis=dict(
        title="평균 선석점유율 (%)",
        range=[0, berth_occupancy_berth_upper],
        ticksuffix="%"
    ),
    barmode="group",
    template="plotly_white",
    autosize=True,
    height=500,
    bargap=0.25,
    legend_title="부두"
)

# 노트북 안에 그래프를 출력합니다.

berth_occupancy_berth_fig.show()


### 마지막 월별 결과 — `berth_occupancy_monthly_result`

기존 마지막 `result` 표의 컬럼 이름과 순서를 유지하고, 변수명만 분석 내용에 맞게 수정.
월을 걸치는 선박은 각 월에 포함된 시간만 나누어 계산.
한 선박이 여러 월에 걸치면 각 월의 `기록수`에 한 번씩 포함.

기존 계산은 전체 기간 표에서 **0시간 기록을 포함**하지만,
마지막 월별 표에서는 **0시간 기록을 제외**합니다.
이 차이도 유지합니다.

In [ ]:
# 4. 원본 마지막 셀과 같이 출항 > 접안인 기록만 월별 계산에 사용합니다.
berth_occupancy_monthly_source = berth_occupancy_detail.loc[
    berth_occupancy_detail["선석종료시각"] > berth_occupancy_detail["접안시각"]
].copy()
berth_occupancy_monthly_frames = []

for berth_occupancy_month_start in MONTHS:
    berth_occupancy_month_end = berth_occupancy_month_start + pd.DateOffset(months=1)

    # 해당 월과 실제로 겹치는 기록만 가져옵니다.
    berth_occupancy_temp = berth_occupancy_monthly_source.loc[
        (berth_occupancy_monthly_source["접안시각"] < berth_occupancy_month_end)
        & (berth_occupancy_monthly_source["선석종료시각"] > berth_occupancy_month_start)
    ].copy()

    if berth_occupancy_temp.empty:
        continue

    berth_occupancy_work_start = berth_occupancy_temp["접안시각"].clip(lower=berth_occupancy_month_start)
    berth_occupancy_work_end = berth_occupancy_temp["선석종료시각"].clip(upper=berth_occupancy_month_end)
    berth_occupancy_temp["작업시간_시간"] = (
        berth_occupancy_work_end - berth_occupancy_work_start
    ).dt.total_seconds() / 3600
    berth_occupancy_temp["기준월"] = berth_occupancy_month_start

    berth_occupancy_monthly_frames.append(
        berth_occupancy_temp[["기준월", "부두", "선석", "작업시간_시간"]].copy()
    )

# 유효한 기록이 하나도 없어도 필요한 컬럼과 자료형을 가진 빈 표를 만듭니다.
berth_occupancy_monthly_detail = berth_occupancy_detail[["부두", "선석", "작업시간_시간"]].head(0).copy()
berth_occupancy_monthly_detail["기준월"] = pd.Series(dtype="datetime64[ns]")
if berth_occupancy_monthly_frames:
    berth_occupancy_monthly_detail = pd.concat(berth_occupancy_monthly_frames, ignore_index=True)

berth_occupancy_monthly_result = (
    berth_occupancy_monthly_detail.groupby(["기준월", "부두", "선석"])
    .agg(
        기록수=("작업시간_시간", "count"),
        총작업시간_시간=("작업시간_시간", "sum"),
    )
)

# 5. 48개월 × 전체 21개 선석을 모두 표시합니다.
# 기록이 없다는 이유만으로 점유율을 0으로 채우지 않습니다.
berth_occupancy_monthly_index = pd.MultiIndex.from_tuples(
    [
        (berth_occupancy_month, berth_occupancy_number, berth_occupancy_berth)
        for berth_occupancy_month in MONTHS
        for berth_occupancy_number, berth_occupancy_berths in EXPECTED_BERTHS.items()
        for berth_occupancy_berth in berth_occupancy_berths
    ],
    names=["기준월", "부두", "선석"],
)
berth_occupancy_monthly_result = berth_occupancy_monthly_result.reindex(berth_occupancy_monthly_index).reset_index()

# 월마다 일수가 다르므로 월별 총시간을 실제 달력 기준으로 계산합니다.
# 윤년인 2024년 2월은 29일 × 24시간 = 696시간입니다.
berth_occupancy_monthly_result["월총시간"] = (
    berth_occupancy_monthly_result["기준월"] + pd.DateOffset(months=1) - berth_occupancy_monthly_result["기준월"]
).dt.total_seconds() / 3600
berth_occupancy_monthly_result["선석점유율_퍼센트"] = (
    berth_occupancy_monthly_result["총작업시간_시간"] / berth_occupancy_monthly_result["월총시간"] * 100
)
berth_occupancy_monthly_result["연도"] = berth_occupancy_monthly_result["기준월"].dt.year
berth_occupancy_monthly_result["월"] = berth_occupancy_monthly_result["기준월"].dt.month

berth_occupancy_monthly_result = berth_occupancy_monthly_result[[
    "연도", "월", "부두", "선석", "기록수",
    "총작업시간_시간", "월총시간", "선석점유율_퍼센트",
]].sort_values(["연도", "월", "부두", "선석"]).reset_index(drop=True)

print("월별·부두별·선석별 선석점유율")
with pd.option_context("display.max_rows", None, "display.precision", 10):
    display(berth_occupancy_monthly_result)